In [7]:
# ===== common imports =====
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

from sklearn.metrics import mean_squared_error, r2_score


## Data Preparation & Feature Engineering

- Load cleaned data from previous steps (after filtering invalid premiums / claims).  
- Focus on *policies with claims* for severity-model; for premium optimization / claim-probability, use full data.  
- Handle missing values via appropriate imputation (numeric: median/mean; categorical: mode or “missing” label).  
- Encode categorical variables (one-hot or similar).  
- (Optional) Create new features — e.g. vehicle age (from RegistrationYear or VehicleIntroDate), value-to-insured ratio, etc.  
- Split data into training and test sets (e.g. 80% train, 20% test) for proper model evaluation.  


In [21]:
# adjust as needed
RAW_PATH = "../data/raw/MachineLearningRating_v3.txt"
# attempt to load; may need to adjust delimiter / parsing depending on file format
try:
    df = pd.read_csv(RAW_PATH, sep="|", engine="python")
    print("Loaded as pipe-delimited file")
except Exception as e:
    print("CSV load failed:", e)
    try:
        df = pd.read_csv(RAW_PATH, sep=r"\s+", engine="python")
        print("Loaded as whitespace-delimited")
    except Exception as e2:
        print("Whitespace load failed:", e2)
        df = pd.read_fwf(RAW_PATH)
        print("Loaded via fixed-width")
df.columns = df.columns.str.strip()

# -----------------------------
# Feature engineering (apply to full dataset)
# -----------------------------
df["VehicleAge"] = 2025 - df["RegistrationYear"]

# For severity model: restrict to those with a claim
df_sev = df[df["TotalClaims"] > 0].copy()
print(df_sev.columns.tolist())

# Select features
numeric_feats = [
    "CustomValueEstimate",
    "SumInsured",
    "kilowatts",
    "cubiccapacity",
    "VehicleAge"
]
categorical_feats = [
    "Province",
    "VehicleType",
    "make",
    "Model",
    "Gender",
    "CoverType"
]

# -----------------------------
# Preprocessing (define but DON'T fit here)
# -----------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_feats),
    ("cat", categorical_transformer, categorical_feats)
])

# -----------------------------
# Severity train/test split
# -----------------------------
X = df_sev[numeric_feats + categorical_feats]
y = df_sev["TotalClaims"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Loaded as pipe-delimited file
['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'VehicleAge']


## Claim-Severity Model (Regression)

We build multiple regression models to predict claim amount (TotalClaims), conditional on a claim occurring.  
We evaluate using **RMSE** (to penalize large errors) and **R²** (for explanatory power).  
We compare three approaches:  
- Linear Regression (baseline)  
- Random Forest Regressor  
- XGBoost Regressor (gradient boosting)  


In [23]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)   # RMSE calculation
    r2 = r2_score(y_test, y_pred)
    return rmse, r2

# NOTE: In this cell, the preprocessor will be fit inside each pipeline on df_sev.
# These models are ONLY for severity benchmarking. Do not reuse them for expected cost later.

# 1. Linear Regression baseline (severity evaluation)
sev_lr = Pipeline([
    ("preproc", preprocessor),
    ("reg", LinearRegression())
])
sev_lr.fit(X_train, y_train)
lr_rmse, lr_r2 = evaluate(sev_lr, X_test, y_test)

# 2. Random Forest (severity evaluation)
sev_rf = Pipeline([
    ("preproc", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
sev_rf.fit(X_train, y_train)
rf_rmse, rf_r2 = evaluate(sev_rf, X_test, y_test)

# 3. XGBoost (severity evaluation)
sev_xgb_eval = Pipeline([
    ("preproc", preprocessor),
    ("reg", xgb.XGBRegressor(random_state=42, n_jobs=-1))
])
sev_xgb_eval.fit(X_train, y_train)
xgb_rmse, xgb_r2 = evaluate(sev_xgb_eval, X_test, y_test)

print("Severity prediction performance:")
print("LinearRegression — RMSE: {:.2f}, R2: {:.3f}".format(lr_rmse, lr_r2))
print("RandomForest     — RMSE: {:.2f}, R2: {:.3f}".format(rf_rmse, rf_r2))
print("XGBoost         — RMSE: {:.2f}, R2: {:.3f}".format(xgb_rmse, xgb_r2))


Severity prediction performance:
LinearRegression — RMSE: 34867.30, R2: 0.244
RandomForest     — RMSE: 38357.06, R2: 0.085
XGBoost         — RMSE: 41104.62, R2: -0.051


## Claim Probability & Premium Estimation

- Build a classification model (e.g. Random Forest or XGBoost) to predict `HasClaim` (binary).  
- Use predicted probability and severity model to estimate an expected claim cost:  
&nbsp;&nbsp;Premium ≈ Probability_of_Claim × Expected_Severity + loading + profit_margin.  
- Evaluate classification model using ROC-AUC, precision/recall (if useful), and compare predicted premium against actuals.  


In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

# -----------------------------
# Classification dataset
# -----------------------------
df_full = df.copy()
df_full["HasClaim"] = (df_full["TotalClaims"] > 0).astype(int)
# VehicleAge already engineered in first cell for df

X_full = df_full[numeric_feats + categorical_feats]
y_full = df_full["HasClaim"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# -----------------------------
# Fit preprocessor ONCE on full dataset for consistent categories
# -----------------------------
preprocessor.fit(df_full[numeric_feats + categorical_feats])

# -----------------------------
# Severity model trained using transformed features from the same preprocessor
# (ensures identical feature space at train and predict time)
# -----------------------------
df_sev_full = df_full[df_full["TotalClaims"] > 0].copy()
X_sev = df_sev_full[numeric_feats + categorical_feats]
y_sev = df_sev_full["TotalClaims"]

# Transform severity training data and classification test data with the SAME preprocessor
X_sev_tr = preprocessor.transform(X_sev)
X_test_c_tr = preprocessor.transform(X_test_c)

sev_reg_xgb = xgb.XGBRegressor(random_state=42, n_jobs=-1)
sev_reg_xgb.fit(X_sev_tr, y_sev)

# -----------------------------
# Claim probability model (can stay as a pipeline using the same preprocessor)
# -----------------------------
clf = Pipeline([
    ("preproc", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
clf.fit(X_train_c, y_train_c)

y_prob = clf.predict_proba(X_test_c)[:, 1]
auc = roc_auc_score(y_test_c, y_prob)
print("Claim probability model ROC-AUC:", auc)
print(classification_report(y_test_c, clf.predict(X_test_c)))

# -----------------------------
# Expected claim cost (pure premium estimate)
# Use the XGB severity model trained on transformed features
# -----------------------------
sev_pred = sev_reg_xgb.predict(X_test_c_tr)
expected_cost = y_prob * sev_pred
print("Expected claim cost sample:", expected_cost[:10])


Claim probability model ROC-AUC: 0.6876760268275786
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    199462
           1       0.00      0.00      0.00       558

    accuracy                           1.00    200020
   macro avg       0.50      0.50      0.50    200020
weighted avg       0.99      1.00      1.00    200020

Expected claim cost sample: [  0.           0.           2.17130138   0.           0.
 165.66132185   0.           0.           0.          34.35444396]
